In [1]:
#Setup
from pathlib import Path
from dotenv import load_dotenv
import os
import sys
import json
import importlib

PROJECT_ROOT = Path(
    "/Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag"
)

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env", override=True)

print("Project:", PROJECT_ROOT)
print("Nebius configured:", bool(os.getenv("NEBIUS_API_KEY")))

Project: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag
Nebius configured: True


In [4]:
import importlib
import retrieval.graph_retriever

importlib.reload(
    retrieval.graph_retriever
)

from retrieval.graph_retriever import (
    retrieve_graph,
    GRAPH_QUERIES,
)

from retrieval.vector_retriever import (
    retrieve_vector,
)

print("Graph intents:", len(GRAPH_QUERIES))

for intent in GRAPH_QUERIES:
    print("-", intent)

print("\nVector retriever ready")

Graph intents: 10
- project_feature_technology
- project_purpose
- decision_rationale
- person_project_skill
- shared_projects
- team_projects
- leader_technologies
- decision_approvers
- expert_lookup
- multi_hop_decisions

Vector retriever ready


In [5]:
#Compare the two retrievers
from retrieval.graph_retriever import (
    retrieve_graph,
    GRAPH_QUERIES,
)

from retrieval.vector_retriever import (
    retrieve_vector,
)

print("Graph intents:", len(GRAPH_QUERIES))
print("Vector retriever ready")

Graph intents: 10
Vector retriever ready


In [6]:
#Configure LLM Planner
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("NEBIUS_API_KEY"),
    base_url="https://api.tokenfactory.nebius.com/v1",
)

PLANNER_MODEL = "Qwen/Qwen3-30B-A3B-Instruct-2507"
ANSWER_MODEL = "Qwen/Qwen3-30B-A3B-Instruct-2507"

print("Planner:", PLANNER_MODEL)

Planner: Qwen/Qwen3-30B-A3B-Instruct-2507


In [7]:
#Load valid entities
manifest = json.loads(
    (
        PROJECT_ROOT
        / "data"
        / "dataset_manifest.json"
    ).read_text()
)

KNOWN_PROJECTS = [
    x["name"]
    for x in manifest["projects"]
]

KNOWN_PEOPLE = [
    x["name"]
    for x in manifest["people"]
]

KNOWN_TEAMS = [
    x["name"]
    for x in manifest["teams"]
]

KNOWN_SKILLS = [
    x["name"]
    for x in manifest["skills"]
]

KNOWN_TECHNOLOGIES = [
    x["name"]
    for x in manifest["technologies"]
]

print("Projects:", KNOWN_PROJECTS)
print("People:", KNOWN_PEOPLE)

Projects: ['Project Atlas', 'Project Phoenix', 'Project Nova', 'Project Mercury']
People: ['Alice Chen', 'Bob Singh', 'Carol Martinez', 'David Kim', 'Elena Rossi', 'Farah Khan', 'George Liu', 'Hannah Brooks', 'Isaac Brown', 'Julia Patel']


In [8]:
#Define planner output schemas
from typing import Literal
from pydantic import BaseModel, Field


GraphIntent = Literal[
    "project_feature_technology",
    "project_purpose",
    "decision_rationale",
    "person_project_skill",
    "shared_projects",
    "decision_approvers",
    "team_projects",
    "leader_technologies",
    "expert_lookup",
    "multi_hop_decisions",
]


class RetrievalPlan(BaseModel):
    strategy: Literal[
        "graph",
        "vector",
        "hybrid",
    ]

    graph_intent: GraphIntent | None = None

    parameters: dict = Field(
        default_factory=dict
    )

    reason: str

In [44]:
import re


def get_required_parameters(query: str) -> list[str]:
    """
    Extract Cypher parameters such as $project, $skill, $keyword.
    """
    return sorted(
        set(
            re.findall(
                r"\$([A-Za-z_][A-Za-z0-9_]*)",
                query
            )
        )
    )


GRAPH_INTENT_PARAMS = {
    intent: get_required_parameters(query)
    for intent, query in GRAPH_QUERIES.items()
}


print("Graph intent parameter requirements:\n")

for intent, params in GRAPH_INTENT_PARAMS.items():
    print(
        f"{intent}: {params}"
    )

Graph intent parameter requirements:

project_feature_technology: ['keyword', 'project']
project_purpose: ['project']
decision_rationale: ['project', 'technology']
person_project_skill: ['project', 'skill']
shared_projects: ['project1', 'project2']
team_projects: ['team']
leader_technologies: ['person']
decision_approvers: ['project']
expert_lookup: ['technology']
multi_hop_decisions: ['project']


In [46]:
#Create LLM Planner
def plan_retrieval(
    question: str,
) -> RetrievalPlan:

    intent_specs = "\n".join(
        (
            f"- {intent}: "
            f"required parameters = "
            f"{GRAPH_INTENT_PARAMS[intent]}"
        )
        for intent in GRAPH_QUERIES.keys()
    )

    prompt = f"""
You are the retrieval planner for an organizational RAG system.

Choose the best retrieval strategy for the user's question.

Available strategies:

GRAPH
Use when the answer primarily requires explicit relationships,
set intersections, ownership, membership, approvals, project
participation, expertise connections, or multi-hop traversal.

VECTOR
Use when the answer primarily requires semantic or narrative
information found in document prose, such as purpose, explanation,
rationale, description, or summary.

HYBRID
Use when both structured relationships AND narrative document
evidence are useful.

AVAILABLE GRAPH INTENTS AND THEIR REQUIRED PARAMETERS:

{intent_specs}

Known projects:
{KNOWN_PROJECTS}

Known people:
{KNOWN_PEOPLE}

Known teams:
{KNOWN_TEAMS}

Known skills:
{KNOWN_SKILLS}

Known technologies:
{KNOWN_TECHNOLOGIES}


EXAMPLES

Question:
Who worked on Project Phoenix and also has Kubernetes experience?

Plan:
{{
    "strategy": "graph",
    "graph_intent": "person_project_skill",
    "parameters": {{
        "project": "Project Phoenix",
        "skill": "Kubernetes"
    }},
    "reason": "Requires intersection of project participation and skill."
}}


Question:
What is the purpose of Project Atlas?

Plan:
{{
    "strategy": "vector",
    "graph_intent": null,
    "parameters": {{}},
    "reason": "Purpose is narrative information."
}}


Question:
What database does Project Phoenix use for its online feature cache?

Plan:
{{
    "strategy": "graph",
    "graph_intent": "project_feature_technology",
    "parameters": {{
        "project": "Project Phoenix",
        "keyword": "online feature cache"
    }},
    "reason": "Requires identifying the technology associated with a specific project feature."
}}


Question:
Who approved the Kafka decision for Project Atlas and why?

Plan:
{{
    "strategy": "hybrid",
    "graph_intent": "decision_approvers",
    "parameters": {{
        "project": "Project Atlas"
    }},
    "reason": "Graph identifies the approver while documents provide rationale."
}}


RULES

1. Never invent an entity.
2. Use human-readable entity names exactly as listed above.
3. GRAPH or HYBRID requires graph_intent.
4. VECTOR requires graph_intent=null and parameters={{}}.
5. When selecting a graph intent, you MUST populate EVERY
   required parameter listed for that intent.
6. Do not omit any required Cypher parameter.
7. For free-text parameters such as "keyword", extract a concise
   phrase directly from the user's question.
8. Do not add unnecessary parameters.
9. Keep reason to one short sentence.

USER QUESTION:
{question}

Return ONLY JSON:

{{
    "strategy": "graph|vector|hybrid",
    "graph_intent": null,
    "parameters": {{}},
    "reason": "..."
}}
"""

    response = client.chat.completions.create(
        model=PLANNER_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0.0,
        max_tokens=450,
        response_format={
            "type": "json_object"
        },
    )

    content = (
        response
        .choices[0]
        .message
        .content
    )

    if not content:
        raise RuntimeError(
            "Planner returned empty output"
        )

    data = json.loads(content)

    plan = RetrievalPlan.model_validate(
        data
    )

    # -----------------------------
    # Validate graph parameters
    # -----------------------------

    if plan.strategy in {
        "graph",
        "hybrid",
    }:

        required = set(
            GRAPH_INTENT_PARAMS[
                plan.graph_intent
            ]
        )

        supplied = set(
            plan.parameters.keys()
        )

        missing = (
            required - supplied
        )

        if missing:
            raise ValueError(
                f"Planner selected "
                f"'{plan.graph_intent}' "
                f"but omitted required "
                f"parameters: "
                f"{sorted(missing)}. "
                f"Generated parameters: "
                f"{plan.parameters}"
            )

    return plan

In [49]:
q1 = (
    "What database does Project Phoenix "
    "use for its online feature cache?"
)

q1_plan = plan_retrieval(q1)

print(
    q1_plan.model_dump()
)

{'strategy': 'graph', 'graph_intent': 'project_feature_technology', 'parameters': {'project': 'Project Phoenix', 'keyword': 'online feature cache'}, 'reason': 'Requires identifying the technology associated with a specific project feature.'}


In [47]:
#Test Planner - Graph
plan_retrieval(
    "Who worked on Project Phoenix "
    "and also has Kubernetes experience?"
)

RetrievalPlan(strategy='graph', graph_intent='person_project_skill', parameters={'project': 'Project Phoenix', 'skill': 'Kubernetes'}, reason='Requires intersection of project participation and skill.')

In [12]:
#Test Planner - Vector
plan_retrieval(
    "What is the purpose of Project Atlas?"
)

RetrievalPlan(strategy='vector', graph_intent=None, parameters={}, reason='The purpose of a project is typically described in narrative form within documents.')

In [13]:
#Hybrid
plan_retrieval(
    "Who approved the Kafka decision "
    "for Project Atlas and why?"
)

RetrievalPlan(strategy='hybrid', graph_intent='decision_approvers', parameters={'project': 'Project Atlas'}, reason='The question requires both approval information and the rationale behind the decision.')

In [14]:
#Format Evidence Types
def format_graph_evidence(results):

    if not results:
        return "No graph evidence retrieved."

    blocks = []

    for i, row in enumerate(
        results,
        start=1,
    ):
        values = " | ".join(
            f"{key}: {value}"
            for key, value in row.items()
            if value is not None
        )

        blocks.append(
            f"Graph evidence {i}: {values}"
        )

    return "\n".join(blocks)


def format_vector_evidence(results):

    if not results:
        return "No vector evidence retrieved."

    blocks = []

    for item in results:

        blocks.append(
            f"""
Source: {item['file_name']}
Similarity: {item['score']:.4f}

{item['text']}
""".strip()
        )

    return "\n\n---\n\n".join(blocks)

In [51]:
#Execute retrieval plan
def execute_plan(
    question: str,
    plan: RetrievalPlan,
):

    graph_results = []
    vector_results = []

    # ------------------------
    # Graph
    # ------------------------

    if plan.strategy in {
        "graph",
        "hybrid",
    }:

        required = set(
            GRAPH_INTENT_PARAMS[
                plan.graph_intent
            ]
        )

        supplied = set(
            plan.parameters.keys()
        )

        missing = (
            required - supplied
        )

        if missing:
            raise ValueError(
                f"Cannot execute graph intent "
                f"'{plan.graph_intent}'. "
                f"Missing parameters: "
                f"{sorted(missing)}"
            )

        graph_response = retrieve_graph(
            intent=plan.graph_intent,
            parameters=plan.parameters,
        )

        graph_results = (
            graph_response["results"]
        )

    # ------------------------
    # Vector
    # ------------------------

    if plan.strategy in {
        "vector",
        "hybrid",
    }:

        vector_response = retrieve_vector(
            question,
            top_k=5,
        )

        vector_results = (
            vector_response["results"]
        )

    return {
        "strategy": plan.strategy,
        "graph_results": graph_results,
        "vector_results": vector_results,
    }

In [52]:
q1_retrieval = execute_plan(
    q1,
    q1_plan,
)

q1_retrieval

{'strategy': 'graph',
 'graph_results': [{'project': 'Project Phoenix',
   'decision': 'Use Redis for Phoenix online feature cache',
   'technology': 'Redis'}],
 'vector_results': []}

In [16]:
#Test planning + retrieval path
question = (
    "Who worked on Project Phoenix "
    "and also has Kubernetes experience?"
)

plan = plan_retrieval(question)

retrieval = execute_plan(
    question,
    plan,
)

print("PLAN")
print(plan.model_dump())

print("\nGRAPH")
for row in retrieval["graph_results"]:
    print(row)

print("\nVECTOR")
for row in retrieval["vector_results"]:
    print(
        row["file_name"],
        row["score"]
    )

PLAN
{'strategy': 'graph', 'graph_intent': 'person_project_skill', 'parameters': {'project': 'Project Phoenix', 'skill': 'Kubernetes'}, 'reason': 'The question asks for people with specific project and skill overlap, which is a direct graph relationship.'}

GRAPH
{'person': 'Bob Singh', 'project': 'Project Phoenix', 'skill': 'Kubernetes'}
{'person': 'Hannah Brooks', 'project': 'Project Phoenix', 'skill': 'Kubernetes'}

VECTOR


In [17]:
#Test Hybrid retrieval
question = (
    "Who approved the Kafka decision "
    "for Project Atlas and why?"
)

plan = plan_retrieval(question)

retrieval = execute_plan(
    question,
    plan,
)

print("PLAN:")
print(plan.model_dump())

print("\nGRAPH EVIDENCE:")
print(
    format_graph_evidence(
        retrieval["graph_results"]
    )
)

print("\nVECTOR EVIDENCE:")
print(
    format_vector_evidence(
        retrieval["vector_results"]
    )[:3000]
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5564.26it/s]
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


PLAN:
{'strategy': 'hybrid', 'graph_intent': 'decision_approvers', 'parameters': {'project': 'Project Atlas'}, 'reason': 'The question requires both approval information and the rationale behind the decision.'}

GRAPH EVIDENCE:
Graph evidence 1: approver: Alice Chen | decision: Adopt Kafka for Atlas change feed | technology: Kafka | project: Project Atlas
Graph evidence 2: approver: Alice Chen | decision: Deploy Atlas ranking services on Kubernetes | technology: Kubernetes | project: Project Atlas
Graph evidence 3: approver: Alice Chen | decision: Standardize Elasticsearch mappings for Atlas | technology: Elasticsearch | project: Project Atlas

VECTOR EVIDENCE:
Source: decision_d001.md
Similarity: 0.9229

---
doc_id: doc_decision_d001
doc_type: architecture_decision
entity_id: decision_d001
project_id: project_atlas
date: 2026-02-12
---
# Adopt Kafka for Atlas change feed

**Decision ID:** decision_d001  
**Project:** Project Atlas  
**Technology:** Kafka  
**Proposed by:** Carol Marti

In [83]:
#Shared final-answer function
AGENT_ANSWER_PROMPT = """
You answer questions about the fictional Acme AI organization.

Use ONLY the retrieved evidence.

EVIDENCE PRIORITY RULES

1. GRAPH EVIDENCE is structured evidence produced by validated
   graph queries. Treat explicit graph rows as authoritative for
   relationships, intersections, memberships, ownership, projects,
   technologies, people, decisions, and multi-hop results.

2. VECTOR EVIDENCE contains narrative document passages. Use it
   primarily for explanations, rationale, purpose, descriptions,
   and supporting context.

3. When Graph and Vector evidence appear to conflict, do NOT use
   Vector evidence to discard explicit structured Graph results.

LIST / SET QUESTIONS

4. When the question asks "which", "who", "what decisions",
   "what projects", or otherwise requests a set/list, inspect ALL
   Graph evidence rows.

5. Include every distinct Graph result that directly satisfies the
   query unless there is explicit evidence that it should be excluded.

6. Do not arbitrarily restrict the answer to only one project, team,
   person, or source when multiple Graph results satisfy the query.

7. Deduplicate repeated rows, but do not omit unique results.

FAITHFULNESS

8. Do not infer relationships that are not explicitly present in the
   retrieved evidence.

9. Never claim that one person worked on a project merely because
   another person on the same team worked on it.

10. Do not invent dates, approvers, proposers, relationships, or
    project participation.

11. If Graph evidence provides the requested structured answer and
    Vector evidence is unnecessary, answer directly from the Graph
    evidence.

12. If both evidence types are useful, combine them without allowing
    narrative evidence to override structured Graph results.

13. If the evidence is genuinely insufficient, say so explicitly.

14. Be concise but complete.
"""


def generate_agent_answer(
    question,
    graph_results,
    vector_results,
):

    graph_context = format_graph_evidence(
        graph_results
    )

    vector_context = format_vector_evidence(
        vector_results
    )

    prompt = f"""
QUESTION:
{question}

STRUCTURED GRAPH RESULT COUNT:
{len(graph_results)}

GRAPH EVIDENCE:
{graph_context}

VECTOR EVIDENCE:
{vector_context}

IMPORTANT:
If the question requests a list/set and the Graph evidence contains
multiple unique matching rows, include every relevant unique result.
Do not silently drop Graph rows because Vector evidence focuses on
only part of the answer.

ANSWER:
"""

    response = client.chat.completions.create(
        model=ANSWER_MODEL,
        messages=[
            {
                "role": "system",
                "content": AGENT_ANSWER_PROMPT,
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=0.0,
        max_tokens=500,
    )

    content = (
        response
        .choices[0]
        .message
        .content
    )

    if not content:
        raise RuntimeError(
            "Answer model returned no content"
        )

    return content.strip()

In [19]:
#End-to_end agent function
def agentic_rag(question: str):

    # 1. Plan
    plan = plan_retrieval(
        question
    )

    # 2. Retrieve
    retrieval = execute_plan(
        question,
        plan,
    )

    # 3. Generate
    answer = generate_agent_answer(
        question=question,
        graph_results=
            retrieval["graph_results"],
        vector_results=
            retrieval["vector_results"],
    )

    return {
        "question": question,
        "strategy": plan.strategy,
        "graph_intent":
            plan.graph_intent,
        "parameters":
            plan.parameters,
        "reason":
            plan.reason,
        "graph_results":
            retrieval["graph_results"],
        "vector_results":
            retrieval["vector_results"],
        "answer":
            answer,
    }

In [20]:
#End-to-end test
result = agentic_rag(
    "Who worked on Project Phoenix "
    "and also has Kubernetes experience?"
)

print("Strategy:", result["strategy"])
print("Reason:", result["reason"])

print("\nAnswer:")
print(result["answer"])

Strategy: graph
Reason: The question asks for individuals with specific project and skill attributes, which is a direct match for the person_project_skill graph intent.

Answer:
Bob Singh and Hannah Brooks worked on Project Phoenix and have Kubernetes experience.


In [21]:
#Test 2
result = agentic_rag(
    "Who approved the Kafka decision "
    "for Project Atlas and why?"
)

print("Strategy:", result["strategy"])
print("Reason:", result["reason"])

print("\nAnswer:")
print(result["answer"])

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


Strategy: hybrid
Reason: The question requires both approval information and the rationale behind the decision.

Answer:
Alice Chen approved the Kafka decision for Project Atlas on February 12, 2026. She approved it because Kafka provided durable event history, replayable and ordered product and behavior updates, and decoupled producers from ranking consumers—meeting the project’s need for reliable, scalable event streaming.


In [22]:
#Evidence-grading schema
from pydantic import BaseModel
from typing import Literal


class EvidenceGrade(BaseModel):
    sufficient: bool

    missing_information: str

    recommended_strategy: Literal[
        "graph",
        "vector",
        "hybrid",
        "none",
    ]

    reason: str

In [23]:
#Evidence grader
#The grader sees the question and retrieved evidence, but not the expected answer. 
# That is important—we don't want evaluation ground truth leaking into our agent.
def grade_evidence(
    question: str,
    graph_results: list,
    vector_results: list,
) -> EvidenceGrade:

    graph_context = format_graph_evidence(
        graph_results
    )

    vector_context = format_vector_evidence(
        vector_results
    )

    prompt = f"""
You are an evidence grader for an organizational RAG system.

Your job is to determine whether the retrieved evidence is sufficient
to answer the user's question completely and faithfully.

QUESTION:
{question}

GRAPH EVIDENCE:
{graph_context}

VECTOR EVIDENCE:
{vector_context}

Evaluate ONLY the evidence shown above.

Rules:

1. Do not answer the user's question.
2. Do not use outside knowledge.
3. Mark sufficient=true only when the evidence supports a complete answer.
4. For list questions, the evidence should support the required set of items,
   not merely one example.
5. For "why", "purpose", "explain", or rationale questions, narrative evidence
   is usually required.
6. For relationship/intersection/multi-hop questions, structured graph
   evidence may be sufficient.
7. If evidence is insufficient, recommend the best next retrieval strategy:
      graph  = structured relationships are missing
      vector = narrative/details are missing
      hybrid = both types would help
8. If evidence is sufficient, recommended_strategy must be "none".

Return ONLY JSON:

{{
    "sufficient": true,
    "missing_information": "",
    "recommended_strategy": "none",
    "reason": "..."
}}
"""

    response = client.chat.completions.create(
        model=PLANNER_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0.0,
        max_tokens=400,
        response_format={
            "type": "json_object"
        },
    )

    content = (
        response
        .choices[0]
        .message
        .content
    )

    if not content:
        raise RuntimeError(
            "Evidence grader returned empty output"
        )

    data = json.loads(content)

    return EvidenceGrade.model_validate(data)

In [24]:
#Test the grader on your Graph test
question = (
    "Who worked on Project Phoenix "
    "and also has Kubernetes experience?"
)

plan = plan_retrieval(question)

retrieval = execute_plan(
    question,
    plan,
)

grade = grade_evidence(
    question=question,
    graph_results=retrieval["graph_results"],
    vector_results=retrieval["vector_results"],
)

print("Plan:")
print(plan.model_dump())

print("\nGrade:")
print(grade.model_dump())

Plan:
{'strategy': 'graph', 'graph_intent': 'person_project_skill', 'parameters': {'project': 'Project Phoenix', 'skill': 'Kubernetes'}, 'reason': 'The question asks for people with specific project and skill relationships.'}

Grade:
{'sufficient': True, 'missing_information': '', 'recommended_strategy': 'none', 'reason': 'The graph evidence includes two individuals, Bob Singh and Hannah Brooks, both of whom are explicitly linked to Project Phoenix and have Kubernetes experience. This fully satisfies the query, which asks for individuals who worked on Project Phoenix and have Kubernetes experience. The evidence provides a complete set of relevant individuals, meeting the requirements for a list question.'}


In [25]:
#Test grader on Hybrid test
question = (
    "Who approved the Kafka decision "
    "for Project Atlas and why?"
)

plan = plan_retrieval(question)

retrieval = execute_plan(
    question,
    plan,
)

grade = grade_evidence(
    question=question,
    graph_results=retrieval["graph_results"],
    vector_results=retrieval["vector_results"],
)

print("Plan:")
print(plan.model_dump())

print("\nGrade:")
print(grade.model_dump())

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


Plan:
{'strategy': 'hybrid', 'graph_intent': 'decision_approvers', 'parameters': {'project': 'Project Atlas'}, 'reason': 'The question requires both approval information and the rationale behind the decision.'}

Grade:
{'sufficient': True, 'missing_information': '', 'recommended_strategy': 'none', 'reason': "The evidence clearly identifies Alice Chen as the approver of the Kafka decision for Project Atlas, as confirmed in both the graph evidence and the vector evidence (decision_d001.md and meeting_m001.md). The rationale for the decision is fully provided in the 'Rationale' section of decision_d001.md, which states that Kafka was adopted because it provides durable event history and decouples producers from ranking consumers, enabling replayable and ordered product and behavior updates. This directly answers both the 'who' and 'why' components of the question. All required information is present and consistent across multiple sources."}


In [27]:
#Create a replanner. This is what makes the workflow more than a single-shot router.
#If the grader says evidence is insufficient, another LLM decision is made using the failure information.
def replan_retrieval(
    question: str,
    previous_plan: RetrievalPlan,
    grade: EvidenceGrade,
) -> RetrievalPlan:

    intents = "\n".join(
        f"- {intent}"
        for intent in GRAPH_QUERIES.keys()
    )

    prompt = f"""
You are replanning retrieval because the first retrieval attempt
did not provide sufficient evidence.

USER QUESTION:
{question}

PREVIOUS PLAN:
{json.dumps(previous_plan.model_dump(), indent=2)}

EVIDENCE GRADER:
{json.dumps(grade.model_dump(), indent=2)}

Available graph intents:
{intents}

Known projects:
{KNOWN_PROJECTS}

Known people:
{KNOWN_PEOPLE}

Known teams:
{KNOWN_TEAMS}

Known skills:
{KNOWN_SKILLS}

Known technologies:
{KNOWN_TECHNOLOGIES}

Choose a better retrieval plan.

Guidelines:

1. Address the missing information identified by the grader.
2. Prefer expanding to HYBRID when the first strategy captured only
   one type of evidence.
3. Do not repeat the exact same failed plan.
4. Never invent entities.
5. Use entity names exactly as provided.
6. GRAPH or HYBRID requires a valid graph_intent.
7. VECTOR should have graph_intent=null.
8. parameters must match the graph intent.
9. Keep reason short.

Return ONLY JSON:

{{
    "strategy": "graph|vector|hybrid",
    "graph_intent": null,
    "parameters": {{}},
    "reason": "..."
}}
"""

    response = client.chat.completions.create(
        model=PLANNER_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0.0,
        max_tokens=450,
        response_format={
            "type": "json_object"
        },
    )

    content = (
        response
        .choices[0]
        .message
        .content
    )

    if not content:
        raise RuntimeError(
            "Replanner returned empty output"
        )

    data = json.loads(content)

    return RetrievalPlan.model_validate(data)

In [28]:
#Test the replanning logic artificially
test_question = (
    "Who approved the Kafka decision "
    "for Project Atlas and why?"
)

fake_previous_plan = RetrievalPlan(
    strategy="graph",
    graph_intent="decision_approvers",
    parameters={
        "project": "Project Atlas"
    },
    reason="Find the approver using graph relationships."
)

fake_grade = EvidenceGrade(
    sufficient=False,
    missing_information=(
        "The graph identifies the approver "
        "but does not explain the rationale."
    ),
    recommended_strategy="hybrid",
    reason=(
        "Narrative evidence is needed to explain why."
    ),
)

new_plan = replan_retrieval(
    question=test_question,
    previous_plan=fake_previous_plan,
    grade=fake_grade,
)

new_plan

RetrievalPlan(strategy='hybrid', graph_intent='decision_rationale', parameters={'project': 'Project Atlas', 'decision': 'Kafka decision'}, reason='To obtain the rationale behind the Kafka decision approval, combining graph relationships with narrative evidence.')

In [29]:
#Agent state for LangGraph
#Now we introduce actual LangGraph.
from typing import TypedDict, Optional, Any


class AgentState(TypedDict, total=False):

    question: str

    plan: dict

    graph_results: list

    vector_results: list

    grade: dict

    attempts: int

    answer: str

    route_history: list

    grader_history: list

In [30]:
#Import LangGraph
from langgraph.graph import (
    StateGraph,
    START,
    END,
)

print("✅ LangGraph imported")

✅ LangGraph imported


In [31]:
#Planner Node
def planner_node(
    state: AgentState,
):

    question = state["question"]

    plan = plan_retrieval(
        question
    )

    return {
        "plan": plan.model_dump(),
        "attempts": 1,
        "route_history": [
            {
                "attempt": 1,
                **plan.model_dump(),
            }
        ],
        "grader_history": [],
    }

In [32]:
#Retrieval Node
def retrieval_node(
    state: AgentState,
):

    question = state["question"]

    plan = RetrievalPlan.model_validate(
        state["plan"]
    )

    retrieval = execute_plan(
        question=question,
        plan=plan,
    )

    return {
        "graph_results":
            retrieval["graph_results"],

        "vector_results":
            retrieval["vector_results"],
    }

In [33]:
#Evidence Grader Node
def grader_node(
    state: AgentState,
):

    grade = grade_evidence(
        question=state["question"],
        graph_results=state.get(
            "graph_results",
            [],
        ),
        vector_results=state.get(
            "vector_results",
            [],
        ),
    )

    history = list(
        state.get(
            "grader_history",
            [],
        )
    )

    history.append({
        "attempt":
            state.get("attempts", 1),
        **grade.model_dump(),
    })

    return {
        "grade": grade.model_dump(),
        "grader_history": history,
    }

In [34]:
#Conditional decision after grading
#We need to prevent an endless agent loop.
MAX_RETRIEVAL_ATTEMPTS = 2


def after_grading(
    state: AgentState,
):

    grade = EvidenceGrade.model_validate(
        state["grade"]
    )

    attempts = state.get(
        "attempts",
        1,
    )

    if grade.sufficient:
        return "answer"

    if attempts >= MAX_RETRIEVAL_ATTEMPTS:
        return "answer"

    return "replan"

In [ ]:
#For a zero-result retrieval, we don't really need an LLM to tell us the exact same tool failed.
def retrieval_is_empty(
    state: AgentState,
) -> bool:

    graph_results = state.get(
        "graph_results",
        []
    )

    vector_results = state.get(
        "vector_results",
        []
    )

    plan = RetrievalPlan.model_validate(
        state["plan"]
    )

    if plan.strategy == "graph":
        return len(graph_results) == 0

    if plan.strategy == "vector":
        return len(vector_results) == 0

    if plan.strategy == "hybrid":
        return (
            len(graph_results) == 0
            and len(vector_results) == 0
        )

    return False

In [121]:
# Replanning Node
def replanner_node(
    state: AgentState,
):

    previous_plan = (
        RetrievalPlan.model_validate(
            state["plan"]
        )
    )

    grade = (
        EvidenceGrade.model_validate(
            state["grade"]
        )
    )

    # --------------------------------------------------
    # Deterministic fallback:
    # If graph returned zero results, broaden to HYBRID
    # --------------------------------------------------

    if (
        previous_plan.strategy == "graph"
        and retrieval_is_empty(state)
    ):

        new_plan = RetrievalPlan(
            strategy="hybrid",
            graph_intent=
                previous_plan.graph_intent,
            parameters=
                previous_plan.parameters,
            reason=(
                "Graph retrieval returned zero results; "
                "adding vector retrieval as fallback."
            ),
        )

    else:

        new_plan = replan_retrieval(
            question=state["question"],
            previous_plan=previous_plan,
            grade=grade,
        )

    # --------------------------------------------------
    # Guardrail:
    # Never repeat the exact same failed retrieval plan
    # --------------------------------------------------

    same_plan = (
        new_plan.strategy
        == previous_plan.strategy

        and new_plan.graph_intent
        == previous_plan.graph_intent

        and new_plan.parameters
        == previous_plan.parameters
    )

    if same_plan:

        # Graph failed → add semantic retrieval
        if previous_plan.strategy == "graph":

            new_plan = RetrievalPlan(
                strategy="hybrid",
                graph_intent=
                    previous_plan.graph_intent,
                parameters=
                    previous_plan.parameters,
                reason=(
                    "The previous graph retrieval "
                    "returned insufficient evidence, "
                    "so vector evidence is added."
                ),
            )

        # Vector failed
        elif previous_plan.strategy == "vector":

            if new_plan.graph_intent is not None:

                new_plan = RetrievalPlan(
                    strategy="hybrid",
                    graph_intent=
                        new_plan.graph_intent,
                    parameters=
                        new_plan.parameters,
                    reason=(
                        "The previous vector retrieval "
                        "returned insufficient evidence, "
                        "so graph evidence is added."
                    ),
                )

        # Hybrid already failed
        elif previous_plan.strategy == "hybrid":

            new_plan = previous_plan

    # --------------------------------------------------
    # Update attempt counter
    # --------------------------------------------------

    new_attempt = (
        state.get(
            "attempts",
            1,
        )
        + 1
    )

    # --------------------------------------------------
    # Update route history
    # --------------------------------------------------

    history = list(
        state.get(
            "route_history",
            [],
        )
    )

    history.append({
        "attempt": new_attempt,
        **new_plan.model_dump(),
    })

    return {
        "plan":
            new_plan.model_dump(),

        "attempts":
            new_attempt,

        "route_history":
            history,
    }

In [36]:
#Final-answer Node
def answer_node(
    state: AgentState,
):

    answer = generate_agent_answer(
        question=state["question"],

        graph_results=state.get(
            "graph_results",
            [],
        ),

        vector_results=state.get(
            "vector_results",
            [],
        ),
    )

    return {
        "answer": answer
    }

In [122]:
#Build the LangGraph workflow
#Nodes
workflow = StateGraph(
    AgentState
)

workflow.add_node(
    "planner",
    planner_node,
)

workflow.add_node(
    "retrieve",
    retrieval_node,
)

workflow.add_node(
    "grader",
    grader_node,
)

workflow.add_node(
    "replan",
    replanner_node,
)

workflow.add_node(
    "answer",
    answer_node,
)

In [124]:
#Edges
workflow.add_edge(
    START,
    "planner",
)

workflow.add_edge(
    "planner",
    "retrieve",
)

workflow.add_edge(
    "retrieve",
    "grader",
)

workflow.add_conditional_edges(
    "grader",
    after_grading,
    {
        "answer": "answer",
        "replan": "replan",
    },
)

workflow.add_edge(
    "replan",
    "retrieve",
)

workflow.add_edge(
    "answer",
    END,
)

In [125]:
#Compile
agent_graph = workflow.compile()

print(
    "✅ Agentic LangGraph workflow compiled"
)

✅ Agentic LangGraph workflow compiled


In [126]:
#First LangGraph Test
result = agent_graph.invoke({
    "question": (
        "Who worked on Project Phoenix "
        "and also has Kubernetes experience?"
    )
})

print("ANSWER:")
print(result["answer"])

print("\nATTEMPTS:")
print(result["attempts"])

print("\nROUTE HISTORY:")
for item in result["route_history"]:
    print(item)

print("\nGRADER HISTORY:")
for item in result["grader_history"]:
    print(item)

ANSWER:
Bob Singh and Hannah Brooks worked on Project Phoenix and have Kubernetes experience.

ATTEMPTS:
1

ROUTE HISTORY:
{'attempt': 1, 'strategy': 'graph', 'graph_intent': 'person_project_skill', 'parameters': {'project': 'Project Phoenix', 'skill': 'Kubernetes'}, 'reason': 'Requires intersection of project participation and skill.'}

GRADER HISTORY:
{'attempt': 1, 'sufficient': True, 'missing_information': '', 'recommended_strategy': 'none', 'reason': 'The graph evidence includes two individuals, Bob Singh and Hannah Brooks, both of whom are linked to Project Phoenix and have Kubernetes experience. This fully satisfies the query, which asks for individuals who worked on Project Phoenix and have Kubernetes experience. The evidence provides a complete set of relevant individuals, meeting the requirements for a list question.'}


In [41]:
#Hybrid LangGraph Test
result = agent_graph.invoke({
    "question": (
        "Who approved the Kafka decision "
        "for Project Atlas and why?"
    )
})

print("ANSWER:")
print(result["answer"])

print("\nATTEMPTS:", result["attempts"])

print("\nROUTE HISTORY:")
for item in result["route_history"]:
    print(item)

print("\nGRADER HISTORY:")
for item in result["grader_history"]:
    print(item)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


ANSWER:
Alice Chen approved the Kafka decision for Project Atlas on February 12, 2026. She approved it because Kafka provided durable event history, replayable and ordered product and behavior updates, and decoupled producers from ranking consumers—meeting the project’s need for reliable, scalable event streaming.

ATTEMPTS: 1

ROUTE HISTORY:
{'attempt': 1, 'strategy': 'hybrid', 'graph_intent': 'decision_approvers', 'parameters': {'project': 'Project Atlas'}, 'reason': 'The question requires both approval information and the rationale behind the decision.'}

GRADER HISTORY:
{'attempt': 1, 'sufficient': True, 'missing_information': '', 'recommended_strategy': 'none', 'reason': 'The evidence clearly identifies Alice Chen as the approver of the Kafka decision for Project Atlas, as stated in both the graph evidence and the vector evidence (decision_d001.md and meeting_m001.md). The rationale provided in decision_d001.md explains that Kafka was adopted because Atlas needed replayable, order

What we have after this - 

                     ┌─────────────┐
                     │   Question  │
                     └──────┬──────┘
                            ↓
                     ┌─────────────┐
                     │ LLM Planner │
                     └──────┬──────┘
                            ↓
                     ┌─────────────┐
                     │  Retrieval  │
                     │ G / V / H   │
                     └──────┬──────┘
                            ↓
                     ┌─────────────┐
                     │ LLM Grader  │
                     └──────┬──────┘
                            ↓
                      Sufficient?
                       /        \
                     Yes         No
                      ↓           ↓
                   Answer      Replanner
                                  ↓
                              Retrieval
                                  ↓
                                Grader
                                  ↓
                                Answer

Evaluate Using the Agent

In [42]:
#Load all 10 evaluation questions
questions_path = (
    PROJECT_ROOT
    / "evaluation"
    / "questions.json"
)

evaluation_questions = json.loads(
    questions_path.read_text()
)

print("Questions:", len(evaluation_questions))

for q in evaluation_questions:
    print(
        f"Q{q['id']}:",
        q["question"]
    )

Questions: 10
Q1: What database does Project Phoenix use for its online feature cache?
Q2: What is the purpose of Project Atlas?
Q3: Why did Project Atlas adopt Kafka?
Q4: Who worked on Project Phoenix and also has Kubernetes experience?
Q5: Which people worked on both Project Atlas and Project Phoenix?
Q6: Who approved the architecture decisions for technologies used by Project Atlas?
Q7: Which projects were worked on by members of the Search team?
Q8: Which technologies are used by projects led by Alice Chen?
Q9: Who is the best person to consult about Kafka based on both project experience and documented technical decisions?
Q10: Which architecture decisions affected projects owned by teams whose members also worked on Project Phoenix?


In [127]:
#Run Agentic RAG on all evaluation questions
import time

agentic_results = []

for item in evaluation_questions:

    print("=" * 100)
    print(
        f"Q{item['id']}: "
        f"{item['question']}"
    )

    start = time.perf_counter()

    try:

        result = agent_graph.invoke({
            "question": item["question"]
        })

        latency = (
            time.perf_counter()
            - start
        )

        record = {
            "id": item["id"],
            "category": item["category"],
            "question": item["question"],
            "expected_answer":
                item["expected_answer"],

            "answer":
                result["answer"],

            "latency_seconds":
                latency,

            "attempts":
                result.get(
                    "attempts",
                    1,
                ),

            "final_plan":
                result.get(
                    "plan"
                ),

            "route_history":
                result.get(
                    "route_history",
                    []
                ),

            "grader_history":
                result.get(
                    "grader_history",
                    []
                ),

            "graph_results":
                result.get(
                    "graph_results",
                    []
                ),

            "vector_results":
                result.get(
                    "vector_results",
                    []
                ),

            "error": None,
        }

        print(
            "Answer:",
            record["answer"]
        )

        print(
            "Attempts:",
            record["attempts"]
        )

        print(
            "Routes:",
            [
                x["strategy"]
                for x in record[
                    "route_history"
                ]
            ]
        )

    except Exception as e:

        latency = (
            time.perf_counter()
            - start
        )

        record = {
            "id": item["id"],
            "category": item["category"],
            "question": item["question"],
            "expected_answer":
                item["expected_answer"],

            "answer": None,
            "latency_seconds":
                latency,
            "attempts": None,
            "final_plan": None,
            "route_history": [],
            "grader_history": [],
            "graph_results": [],
            "vector_results": [],
            "error": str(e),
        }

        print(
            "❌ ERROR:",
            e
        )

    agentic_results.append(
        record
    )

Q1: What database does Project Phoenix use for its online feature cache?
Answer: Project Phoenix uses Redis for its online feature cache.
Attempts: 1
Routes: ['graph']
Q2: What is the purpose of Project Atlas?


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


Answer: The purpose of Project Atlas is to modernize product search ranking by incorporating fresh behavioral signals and establishing a unified retrieval stack.
Attempts: 1
Routes: ['vector']
Q3: Why did Project Atlas adopt Kafka?


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


Answer: Project Atlas adopted Kafka to enable replayable, ordered product and behavior updates. Kafka provided a durable event history and decoupled producers from ranking consumers, addressing the need for reliable and scalable change feeds. This decision was formally documented in architecture decision *decision_d001*, proposed by Carol Martinez and approved by Alice Chen on 2026-02-12. The adoption aligns with Acme AI’s streaming platform standard, which recommends Kafka for use cases requiring replay, ordered partitions, and multiple independent consumers.
Attempts: 1
Routes: ['hybrid']
Q4: Who worked on Project Phoenix and also has Kubernetes experience?
Answer: Bob Singh and Hannah Brooks worked on Project Phoenix and have Kubernetes experience.
Attempts: 1
Routes: ['graph']
Q5: Which people worked on both Project Atlas and Project Phoenix?
Answer: The people who worked on both Project Atlas and Project Phoenix are:

- Alice Chen  
- Bob Singh  
- Carol Martinez  
- Hannah Brooks

...

Answer: The technologies used by projects led by Alice Chen are:

- Kafka  
- Elasticsearch  
- Python  
- Kubernetes  

These technologies are used in **Project Atlas**, which is led by Alice Chen. This is confirmed by multiple sources, including the project overview, technical architecture document, and project ownership records. Project Phoenix is not led by Alice Chen (it is led by Farah Khan), so its technologies (Kafka, Kubernetes, Redis, Python) are not included in this answer.
Attempts: 2
Routes: ['graph', 'hybrid']
Q9: Who is the best person to consult about Kafka based on both project experience and documented technical decisions?


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


Answer: The best person to consult about Kafka based on both project experience and documented technical decisions is **Carol Martinez**.

### Justification:
- **Project Experience**: Carol Martinez has worked on **3 projects** involving Kafka: *Project Atlas*, *Project Phoenix*, and *Project Mercury*.
- **Documented Technical Decisions**: She is the **proposer** of two key Kafka decisions:
  - *Adopt Kafka for Atlas change feed* (decision_d001)
  - *Use Kafka as Phoenix behavior event bus* (decision_d005)
- **Authority**: According to the *streaming_platform_standard.md* document, Carol Martinez is the **primary technical contact for Kafka architecture** and maintains the organization-wide streaming guidance.
- **Additional Support**: The *recommendation_architecture.md* document confirms she proposed the Kafka decision for Project Phoenix, and the *meeting_m001.md* notes confirm her role in proposing the Kafka adoption for Project Atlas.

No other individual has both a higher project

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


Answer: The architecture decisions that affected projects owned by teams whose members also worked on Project Phoenix are:

1. **Use Kafka as Phoenix behavior event bus** – Project: Project Phoenix, Team: Recommendations, Connected people: Farah Khan, Bob Singh  
2. **Use Redis for Phoenix online feature cache** – Project: Project Phoenix, Team: Recommendations, Connected people: Farah Khan, Bob Singh  
3. **Adopt Kafka for Atlas change feed** – Project: Project Atlas, Team: Search, Connected people: Alice Chen  
4. **Deploy Atlas ranking services on Kubernetes** – Project: Project Atlas, Team: Search, Connected people: Alice Chen  
5. **Standardize Elasticsearch mappings for Atlas** – Project: Project Atlas, Team: Search, Connected people: Alice Chen  
6. **Use Airflow for Mercury settlement workflows** – Project: Project Mercury, Team: Payments, Connected people: Julia Patel  
7. **Use PostgreSQL as Mercury reconciliation ledger** – Project: Project Mercury, Team: Payments, Connected

In [92]:
#Check if any failures
failed = [
    x
    for x in agentic_results
    if x["error"] is not None
]

print(
    "Successful:",
    len(agentic_results)
    - len(failed)
)

print(
    "Failed:",
    len(failed)
)

for x in failed:
    print(
        f"Q{x['id']}:",
        x["error"]
    )

Successful: 10
Failed: 0


In [128]:
#Inspect how the agent routed the questions
for x in agentic_results:

    routes = [
        r["strategy"]
        for r in x["route_history"]
    ]

    print(
        f"Q{x['id']} | "
        f"Attempts={x['attempts']} | "
        f"Routes={routes}"
    )

Q1 | Attempts=1 | Routes=['graph']
Q2 | Attempts=1 | Routes=['vector']
Q3 | Attempts=1 | Routes=['hybrid']
Q4 | Attempts=1 | Routes=['graph']
Q5 | Attempts=1 | Routes=['graph']
Q6 | Attempts=1 | Routes=['hybrid']
Q7 | Attempts=1 | Routes=['graph']
Q8 | Attempts=2 | Routes=['graph', 'hybrid']
Q9 | Attempts=1 | Routes=['hybrid']
Q10 | Attempts=1 | Routes=['hybrid']


In [129]:
#Route distribution
from collections import Counter

initial_routes = Counter(
    x["route_history"][0]["strategy"]
    for x in agentic_results
    if x["route_history"]
)

print(
    "Initial route distribution:"
)

for route, count in (
    initial_routes.items()
):
    print(
        route,
        count
    )

Initial route distribution:
graph 5
vector 1
hybrid 4


In [95]:
#Check retries
retry_count = sum(
    1
    for x in agentic_results
    if (
        x["attempts"]
        is not None
        and x["attempts"] > 1
    )
)

print(
    "Questions requiring replanning:",
    retry_count,
)

Questions requiring replanning: 1


In [96]:
#Inspect replanned questions
for x in agentic_results:

    if (
        x["attempts"]
        and x["attempts"] > 1
    ):

        print("=" * 100)

        print(
            f"Q{x['id']}:",
            x["question"]
        )

        print(
            "\nROUTE HISTORY"
        )

        for route in (
            x["route_history"]
        ):
            print(route)

        print(
            "\nGRADER HISTORY"
        )

        for grade in (
            x["grader_history"]
        ):
            print(grade)

        print(
            "\nFINAL ANSWER"
        )

        print(
            x["answer"]
        )

Q8: Which technologies are used by projects led by Alice Chen?

ROUTE HISTORY
{'attempt': 1, 'strategy': 'graph', 'graph_intent': 'leader_technologies', 'parameters': {'person': 'Alice Chen'}, 'reason': 'Requires identifying technologies used by projects led by a specific person.'}
{'attempt': 2, 'strategy': 'graph', 'graph_intent': 'leader_technologies', 'parameters': {'person': 'Alice Chen'}, 'reason': 'Previous attempt failed due to no evidence; this plan directly targets the relationship between Alice Chen and the technologies used in her projects using the correct graph intent.'}

GRADER HISTORY
{'attempt': 1, 'sufficient': False, 'missing_information': 'No evidence retrieved to answer the question about technologies used by projects led by Alice Chen.', 'recommended_strategy': 'graph', 'reason': 'The question asks for a list of technologies used by projects led by Alice Chen, which requires specific information about project details and associated technologies. No graph or vector

In [130]:
#Save Raw Agentic RAG Results
agentic_output_path = (
    PROJECT_ROOT
    / "evaluation"
    / "agentic_rag_results.json"
)

agentic_output_path.write_text(
    json.dumps(
        agentic_results,
        indent=2,
        default=str,
    )
)

print(
    "✅ Saved:",
    agentic_output_path
)

✅ Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/agentic_rag_results.json


In [131]:
#Format all the gathered evidence by agent
def format_agent_evidence(
    item
):

    graph_context = (
        format_graph_evidence(
            item.get(
                "graph_results",
                []
            )
        )
    )

    vector_context = (
        format_vector_evidence(
            item.get(
                "vector_results",
                []
            )
        )
    )

    return f"""
GRAPH EVIDENCE:
{graph_context}

VECTOR EVIDENCE:
{vector_context}
""".strip()

In [132]:
#Define the LLM judge
JUDGE_MODEL = (
    "openai/gpt-oss-120b"
)


def judge_answer(
    question,
    expected_answer,
    evidence,
    answer,
):

    prompt = f"""
You are evaluating a RAG system.

QUESTION:
{question}

EXPECTED ANSWER:
{json.dumps(
    expected_answer,
    indent=2
)}

RETRIEVED EVIDENCE:
{evidence}

GENERATED ANSWER:
{answer}

Score the generated answer on three dimensions.

CORRECTNESS
0 = substantially incorrect
1 = partially correct
2 = correct

COMPLETENESS
0 = misses most required information
1 = contains some but not all required information
2 = includes all required information

FAITHFULNESS
Evaluate only whether claims actually made in the generated
answer are supported by the retrieved evidence.

Do NOT penalize faithfulness for missing information.
Missing required information affects completeness instead.

0 = unsupported or contradicted claims
1 = mixture of supported and unsupported claims
2 = every claim made is supported by the evidence

Return ONLY JSON:

{{
    "correctness": 0,
    "completeness": 0,
    "faithfulness": 0,
    "reason": "brief explanation"
}}
"""

    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=0.0,
        max_tokens=1000,
        response_format={
            "type": "json_object"
        },
    )

    content = (
        response
        .choices[0]
        .message
        .content
    )

    if not content:
        raise RuntimeError(
            "Judge returned no content"
        )

    return json.loads(content)

In [133]:
#Judge all 10 Agentic RAG results
agentic_judged = []

for item in agentic_results:

    print(
        f"Judging Q{item['id']}..."
    )

    if item["error"] is not None:

        agentic_judged.append({
            **item,
            "judge": None,
            "agent_score": 0.0,
        })

        continue

    evidence = (
        format_agent_evidence(
            item
        )
    )

    try:

        judgment = judge_answer(
            question=
                item["question"],

            expected_answer=
                item[
                    "expected_answer"
                ],

            evidence=evidence,

            answer=
                item["answer"],
        )

        score = (
            judgment["correctness"]
            + judgment["completeness"]
            + judgment["faithfulness"]
        ) / 6

    except Exception as e:

        print(
            "Judge error:",
            e
        )

        judgment = None
        score = None

    agentic_judged.append({
        **item,
        "judge": judgment,
        "agent_score": score,
    })

Judging Q1...
Judging Q2...
Judging Q3...
Judging Q4...
Judging Q5...
Judging Q6...
Judging Q7...
Judging Q8...
Judging Q9...
Judging Q10...


In [134]:
#Inspect Agent Scores
for x in agentic_judged:

    print(
        f"Q{x['id']} | "
        f"Score="
        f"{x['agent_score']:.3f}"
        if x["agent_score"]
        is not None
        else
        f"Q{x['id']} | Score=None"
    )

    print(
        x["judge"]
    )

    print()

Q1 | Score=1.000
{'correctness': 2, 'completeness': 2, 'faithfulness': 2, 'reason': 'The answer correctly states that Project Phoenix uses Redis for its online feature cache, which matches the retrieved graph evidence and fully addresses the question.'}

Q2 | Score=1.000
{'correctness': 2, 'completeness': 2, 'faithfulness': 2, 'reason': 'The answer correctly restates the purpose, includes all required elements, and each claim is directly supported by the retrieved evidence.'}

Q3 | Score=1.000
{'correctness': 2, 'completeness': 2, 'faithfulness': 2, 'reason': 'The answer correctly states that Atlas needed replayable, ordered product and behavior updates and wanted to decouple producers from ranking consumers, matching the expected answer. It also includes all required information and adds supported details (decision ID, proposers, approval, and alignment with the streaming platform standard). All claims are backed by the retrieved evidence.'}

Q4 | Score=1.000
{'correctness': 2, 'compl

In [135]:
#Overall Agentic RAG Score
valid_scores = [
    x["agent_score"]
    for x in agentic_judged
    if x["agent_score"]
    is not None
]

agent_average = (
    sum(valid_scores)
    / len(valid_scores)
)

print(
    "Agentic RAG average:",
    round(
        agent_average * 100,
        2
    ),
    "%"
)

Agentic RAG average: 98.33 %


In [136]:
#Save judged Agent results
agentic_judged_path = (
    PROJECT_ROOT
    / "evaluation"
    / "agentic_rag_judged.json"
)

agentic_judged_path.write_text(
    json.dumps(
        agentic_judged,
        indent=2,
        default=str,
    )
)

print(
    "✅ Saved:",
    agentic_judged_path
)

✅ Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/agentic_rag_judged.json


GraphRAG vs VectorRAG vs AgenticRAG

In [137]:
#Load the frozen baseline comparison
import pandas as pd

baseline_path = (
    PROJECT_ROOT
    / "evaluation"
    / "rag_comparison_summary.csv"
)

baseline_df = pd.read_csv(
    baseline_path
)

print("Columns:")
print(baseline_df.columns.tolist())

display(baseline_df)

Columns:
['Q', 'Category', 'Graph Score', 'Vector Score', 'Winner', 'Graph Latency', 'Vector Latency']


,Q,Category,Graph Score,Vector Score,Winner,Graph Latency,Vector Latency
0,1,single_fact,100.0,100.0,Tie,3.49,3.77
1,2,single_fact,100.0,100.0,Tie,0.70,1.35
2,3,semantic,100.0,100.0,Tie,0.95,0.97
3,4,relationship,100.0,100.0,Tie,1.46,0.63
4,5,relationship,83.3,100.0,VectorRAG,0.79,1.30
5,6,multi_hop,100.0,100.0,Tie,0.79,0.74
6,7,relationship,100.0,33.3,GraphRAG,2.21,5.77
7,8,multi_hop,33.3,100.0,VectorRAG,0.44,2.22
8,9,multi_hop,100.0,100.0,Tie,1.65,1.93
9,10,multi_hop,66.7,16.7,GraphRAG,2.24,4.17


In [138]:
#Create Agentic RAG comparison data
agent_rows = []

for item in agentic_judged:

    routes = [
        x["strategy"]
        for x in item.get(
            "route_history",
            []
        )
    ]

    initial_route = (
        routes[0]
        if routes
        else None
    )

    final_route = (
        routes[-1]
        if routes
        else None
    )

    agent_rows.append({
        "Q": item["id"],
        "Agent Score":
            item["agent_score"] * 100
            if item["agent_score"] is not None
            else None,
        "Agent Latency":
            item["latency_seconds"],
        "Initial Route":
            initial_route,
        "Final Route":
            final_route,
        "Attempts":
            item["attempts"],
    })


agent_df = pd.DataFrame(
    agent_rows
)

display(agent_df)

,Q,Agent Score,Agent Latency,Initial Route,Final Route,Attempts
0,1,100.000000,6.622872,graph,graph,1
1,2,100.000000,9.427435,vector,vector,1
2,3,100.000000,21.192445,hybrid,hybrid,1
3,4,100.000000,11.920682,graph,graph,1
4,5,83.333333,12.861832,graph,graph,1
5,6,100.000000,11.159241,hybrid,hybrid,1
6,7,100.000000,8.037959,graph,graph,1
7,8,100.000000,28.032765,graph,hybrid,2
8,9,100.000000,22.194246,hybrid,hybrid,1
9,10,100.000000,22.840806,hybrid,hybrid,1


In [139]:
#Check baseline column names
baseline_df.head()

,Q,Category,Graph Score,Vector Score,Winner,Graph Latency,Vector Latency
0,1,single_fact,100.0,100.0,Tie,3.49,3.77
1,2,single_fact,100.0,100.0,Tie,0.70,1.35
2,3,semantic,100.0,100.0,Tie,0.95,0.97
3,4,relationship,100.0,100.0,Tie,1.46,0.63
4,5,relationship,83.3,100.0,VectorRAG,0.79,1.30


In [140]:
print(
    baseline_df.columns.tolist()
)

['Q', 'Category', 'Graph Score', 'Vector Score', 'Winner', 'Graph Latency', 'Vector Latency']


In [141]:
#Merge all 3 systems
final_comparison = (
    baseline_df
    .merge(
        agent_df,
        on="Q",
        how="left",
    )
)

display(
    final_comparison
)

,Q,Category,Graph Score,Vector Score,Winner,Graph Latency,Vector Latency,Agent Score,Agent Latency,Initial Route,Final Route,Attempts
0,1,single_fact,100.0,100.0,Tie,3.49,3.77,100.000000,6.622872,graph,graph,1
1,2,single_fact,100.0,100.0,Tie,0.70,1.35,100.000000,9.427435,vector,vector,1
2,3,semantic,100.0,100.0,Tie,0.95,0.97,100.000000,21.192445,hybrid,hybrid,1
3,4,relationship,100.0,100.0,Tie,1.46,0.63,100.000000,11.920682,graph,graph,1
4,5,relationship,83.3,100.0,VectorRAG,0.79,1.30,83.333333,12.861832,graph,graph,1
5,6,multi_hop,100.0,100.0,Tie,0.79,0.74,100.000000,11.159241,hybrid,hybrid,1
6,7,relationship,100.0,33.3,GraphRAG,2.21,5.77,100.000000,8.037959,graph,graph,1
7,8,multi_hop,33.3,100.0,VectorRAG,0.44,2.22,100.000000,28.032765,graph,hybrid,2
8,9,multi_hop,100.0,100.0,Tie,1.65,1.93,100.000000,22.194246,hybrid,hybrid,1
9,10,multi_hop,66.7,16.7,GraphRAG,2.24,4.17,100.000000,22.840806,hybrid,hybrid,1


In [142]:
#Determine the 3-way winner
TIE_THRESHOLD = 5.0


def three_way_winner(row):

    scores = {
        "GraphRAG":
            row["Graph Score"],
        "VectorRAG":
            row["Vector Score"],
        "Agentic RAG":
            row["Agent Score"],
    }

    ordered = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True,
    )

    best_name, best_score = (
        ordered[0]
    )

    second_name, second_score = (
        ordered[1]
    )

    if (
        best_score
        - second_score
        <= TIE_THRESHOLD
    ):
        return "Tie"

    return best_name


final_comparison[
    "Three-Way Winner"
] = final_comparison.apply(
    three_way_winner,
    axis=1,
)

display(
    final_comparison[
        [
            "Q",
            "Category",
            "Graph Score",
            "Vector Score",
            "Agent Score",
            "Initial Route",
            "Attempts",
            "Three-Way Winner",
        ]
    ]
)

,Q,Category,Graph Score,Vector Score,Agent Score,Initial Route,Attempts,Three-Way Winner
0,1,single_fact,100.0,100.0,100.000000,graph,1,Tie
1,2,single_fact,100.0,100.0,100.000000,vector,1,Tie
2,3,semantic,100.0,100.0,100.000000,hybrid,1,Tie
3,4,relationship,100.0,100.0,100.000000,graph,1,Tie
4,5,relationship,83.3,100.0,83.333333,graph,1,VectorRAG
5,6,multi_hop,100.0,100.0,100.000000,hybrid,1,Tie
6,7,relationship,100.0,33.3,100.000000,graph,1,Tie
7,8,multi_hop,33.3,100.0,100.000000,graph,2,Tie
8,9,multi_hop,100.0,100.0,100.000000,hybrid,1,Tie
9,10,multi_hop,66.7,16.7,100.000000,hybrid,1,Agentic RAG


In [143]:
#Overall Scores
overall_scores = pd.DataFrame({
    "System": [
        "GraphRAG",
        "VectorRAG",
        "Agentic RAG",
    ],
    "Average Score": [
        final_comparison[
            "Graph Score"
        ].mean(),

        final_comparison[
            "Vector Score"
        ].mean(),

        final_comparison[
            "Agent Score"
        ].mean(),
    ],
})


display(
    overall_scores.round(2)
)

,System,Average Score
0,GraphRAG,88.33
1,VectorRAG,85.00
2,Agentic RAG,98.33


In [144]:
#Winner counts
winner_counts = (
    final_comparison[
        "Three-Way Winner"
    ]
    .value_counts()
)

print(
    winner_counts
)

Three-Way Winner
Tie            8
VectorRAG      1
Agentic RAG    1
Name: count, dtype: int64


In [145]:
#Latency Comparison
latency_summary = pd.DataFrame({
    "System": [
        "GraphRAG",
        "VectorRAG",
        "Agentic RAG",
    ],

    "Average Latency (s)": [
        final_comparison[
            "Graph Latency"
        ].mean(),

        final_comparison[
            "Vector Latency"
        ].mean(),

        final_comparison[
            "Agent Latency"
        ].mean(),
    ],
})


display(
    latency_summary.round(2)
)

,System,Average Latency (s)
0,GraphRAG,1.47
1,VectorRAG,2.28
2,Agentic RAG,15.43


In [146]:
#Analyze how the Agent routed
route_summary = (
    final_comparison[
        "Initial Route"
    ]
    .value_counts()
    .rename_axis(
        "Initial Route"
    )
    .reset_index(
        name="Questions"
    )
)

display(route_summary)

,Initial Route,Questions
0,graph,5
1,hybrid,4
2,vector,1


In [112]:
#Ouestions requiring replanning
print(
    "Questions requiring replanning:",
    (
        final_comparison[
            "Attempts"
        ] > 1
    ).sum()
)

Questions requiring replanning: 1


In [113]:
#Which question had replanning
display(
    final_comparison.loc[
        final_comparison[
            "Attempts"
        ] > 1,
        [
            "Q",
            "Category",
            "Initial Route",
            "Final Route",
            "Attempts",
            "Agent Score",
        ],
    ]
)

,Q,Category,Initial Route,Final Route,Attempts,Agent Score
7,8,multi_hop,graph,graph,2,33.333333


In [147]:
#Category level comparison
category_comparison = (
    final_comparison
    .groupby(
        "Category"
    )
    .agg({
        "Graph Score":
            "mean",

        "Vector Score":
            "mean",

        "Agent Score":
            "mean",
    })
    .round(2)
)

display(
    category_comparison
)

,Graph Score,Vector Score,Agent Score
Category,,,
multi_hop,75.00,79.18,100.00
relationship,94.43,77.77,94.44
semantic,100.00,100.00,100.00
single_fact,100.00,100.00,100.00


In [148]:
#Agent improvement over best static baseline
final_comparison[
    "Best Static Score"
] = final_comparison[
    [
        "Graph Score",
        "Vector Score",
    ]
].max(
    axis=1
)

final_comparison[
    "Agent vs Best Static"
] = (
    final_comparison[
        "Agent Score"
    ]
    -
    final_comparison[
        "Best Static Score"
    ]
)

display(
    final_comparison[
        [
            "Q",
            "Graph Score",
            "Vector Score",
            "Best Static Score",
            "Agent Score",
            "Agent vs Best Static",
            "Initial Route",
            "Attempts",
        ]
    ].round(2)
)

,Q,Graph Score,Vector Score,Best Static Score,Agent Score,Agent vs Best Static,Initial Route,Attempts
0,1,100.0,100.0,100.0,100.00,0.00,graph,1
1,2,100.0,100.0,100.0,100.00,0.00,vector,1
2,3,100.0,100.0,100.0,100.00,0.00,hybrid,1
3,4,100.0,100.0,100.0,100.00,0.00,graph,1
4,5,83.3,100.0,100.0,83.33,-16.67,graph,1
5,6,100.0,100.0,100.0,100.00,0.00,hybrid,1
6,7,100.0,33.3,100.0,100.00,0.00,graph,1
7,8,33.3,100.0,100.0,100.00,0.00,graph,2
8,9,100.0,100.0,100.0,100.00,0.00,hybrid,1
9,10,66.7,16.7,66.7,100.00,33.30,hybrid,1


In [149]:
print(
    "Average Agent improvement "
    "vs best static baseline:",
    round(
        final_comparison[
            "Agent vs Best Static"
        ].mean(),
        2,
    ),
    "percentage points"
)

Average Agent improvement vs best static baseline: 1.66 percentage points


In [117]:
#Investigate why Agentic RAG had lower scores than the best static baseline on Q8
q8_agent = next(
    x for x in agentic_judged
    if x["id"] == 8
)

print("QUESTION:")
print(q8_agent["question"])

print("\nROUTE HISTORY:")
for x in q8_agent["route_history"]:
    print(x)

print("\nGRADER HISTORY:")
for x in q8_agent["grader_history"]:
    print(x)

print("\nGRAPH RESULTS:")
for x in q8_agent["graph_results"]:
    print(x)

print("\nVECTOR RESULTS:")
for x in q8_agent["vector_results"]:
    print(
        x["file_name"],
        x["score"]
    )

print("\nFINAL ANSWER:")
print(q8_agent["answer"])

print("\nJUDGE:")
print(q8_agent["judge"])

QUESTION:
Which technologies are used by projects led by Alice Chen?

ROUTE HISTORY:
{'attempt': 1, 'strategy': 'graph', 'graph_intent': 'leader_technologies', 'parameters': {'person': 'Alice Chen'}, 'reason': 'Requires identifying technologies used by projects led by a specific person.'}
{'attempt': 2, 'strategy': 'graph', 'graph_intent': 'leader_technologies', 'parameters': {'person': 'Alice Chen'}, 'reason': 'Previous attempt failed due to no evidence; this plan directly targets the relationship between Alice Chen and the technologies used in her projects using the correct graph intent.'}

GRADER HISTORY:
{'attempt': 1, 'sufficient': False, 'missing_information': 'No evidence retrieved to answer the question about technologies used by projects led by Alice Chen.', 'recommended_strategy': 'graph', 'reason': 'The question asks for a list of technologies used by projects led by Alice Chen, which requires specific information about project details and associated technologies. No graph o

In [84]:
#Re-run Q10 
q10_result = agent_graph.invoke({
    "question": (
        "Which architecture decisions affected projects owned by "
        "teams whose members also worked on Project Phoenix?"
    )
})

print("PLAN:")
print(q10_result["plan"])

print("\nGRAPH RESULT COUNT:")
print(
    len(
        q10_result["graph_results"]
    )
)

print("\nANSWER:")
print(
    q10_result["answer"]
)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('db.index.vector.queryNodes' has been replaced by 'SEARCH')} {position: line: 1, column: 1, offset: 0} for query: 'CALL db.index.vector.queryNodes($index, $k, $embedding) YIELD node, score RETURN node.`text` AS text, score, node.id AS id, node {.*, `text`: Null, `embedding`: Null, id: Null } AS metadata'


PLAN:
{'strategy': 'hybrid', 'graph_intent': 'multi_hop_decisions', 'parameters': {'project': 'Project Phoenix'}, 'reason': 'Requires tracing decisions linked to projects via team membership and ownership.'}

GRAPH RESULT COUNT:
7

ANSWER:
The architecture decisions that affected projects owned by teams whose members also worked on Project Phoenix are:

1. **Use Kafka as Phoenix behavior event bus** – Project: Project Phoenix, Team: Recommendations, Connected people: Farah Khan, Bob Singh  
2. **Use Redis for Phoenix online feature cache** – Project: Project Phoenix, Team: Recommendations, Connected people: Farah Khan, Bob Singh  
3. **Adopt Kafka for Atlas change feed** – Project: Project Atlas, Team: Search, Connected people: Alice Chen  
4. **Deploy Atlas ranking services on Kubernetes** – Project: Project Atlas, Team: Search, Connected people: Alice Chen  
5. **Standardize Elasticsearch mappings for Atlas** – Project: Project Atlas, Team: Search, Connected people: Alice Chen  
6. *

In [85]:
#Rejudge Q10
q10_evidence = f"""
GRAPH EVIDENCE:
{format_graph_evidence(q10_result["graph_results"])}

VECTOR EVIDENCE:
{format_vector_evidence(q10_result["vector_results"])}
"""

q10_expected = next(
    q["expected_answer"]
    for q in evaluation_questions
    if q["id"] == 10
)

q10_judge = judge_answer(
    question=(
        "Which architecture decisions affected projects owned by "
        "teams whose members also worked on Project Phoenix?"
    ),
    expected_answer=q10_expected,
    evidence=q10_evidence,
    answer=q10_result["answer"],
)

q10_judge

{'correctness': 2,
 'completeness': 2,
 'faithfulness': 2,
 'reason': 'The answer lists exactly the seven decisions required, covering all items (correctness and completeness). All statements about team membership and contributions are backed by the provided graph and document evidence, so the answer is fully faithful.'}

In [ ]:
#recompile LangGraph
agent_graph = workflow.compile()

In [150]:
#Re-test the agentic scores now
#Create Agentic RAG comparison data
agent_rows = []

for item in agentic_judged:

    routes = [
        x["strategy"]
        for x in item.get(
            "route_history",
            []
        )
    ]

    initial_route = (
        routes[0]
        if routes
        else None
    )

    final_route = (
        routes[-1]
        if routes
        else None
    )

    agent_rows.append({
        "Q": item["id"],
        "Agent Score":
            item["agent_score"] * 100
            if item["agent_score"] is not None
            else None,
        "Agent Latency":
            item["latency_seconds"],
        "Initial Route":
            initial_route,
        "Final Route":
            final_route,
        "Attempts":
            item["attempts"],
    })


agent_df = pd.DataFrame(
    agent_rows
)

display(agent_df)

,Q,Agent Score,Agent Latency,Initial Route,Final Route,Attempts
0,1,100.000000,6.622872,graph,graph,1
1,2,100.000000,9.427435,vector,vector,1
2,3,100.000000,21.192445,hybrid,hybrid,1
3,4,100.000000,11.920682,graph,graph,1
4,5,83.333333,12.861832,graph,graph,1
5,6,100.000000,11.159241,hybrid,hybrid,1
6,7,100.000000,8.037959,graph,graph,1
7,8,100.000000,28.032765,graph,hybrid,2
8,9,100.000000,22.194246,hybrid,hybrid,1
9,10,100.000000,22.840806,hybrid,hybrid,1


In [151]:
#Save Final Comparison
final_csv_path = (
    PROJECT_ROOT
    / "evaluation"
    / "final_three_way_comparison.csv"
)

final_comparison.to_csv(
    final_csv_path,
    index=False,
)


final_json_path = (
    PROJECT_ROOT
    / "evaluation"
    / "final_three_way_comparison.json"
)

final_comparison.to_json(
    final_json_path,
    orient="records",
    indent=2,
)

print(
    "✅ Saved:",
    final_csv_path
)

print(
    "✅ Saved:",
    final_json_path
)

✅ Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/final_three_way_comparison.csv
✅ Saved: /Users/rohitk/mastering_agentic_ai/mastering_agentic_ai_projects/Week_2/orgmind-graphrag/evaluation/final_three_way_comparison.json
